# Supplementary Figure 2

Panels a,b show how often authentic and composition-scrambled peptide queries of one healthy-stool acquisition match a protein of the 233.5-million-sequence reservoir, by query length. Panels c,d show how accepted peptides and selected proteins change with the minimum lookup length and the top-scoring fraction (58 acquisitions, 19 arms). Panels e,f (the continuation) show the exploratory bag and mass-substitution lookup comparison (48 acquisitions) and a separate six-acquisition fragment audit.

Run from the repository root: `python paper.py build SFig02`. Panels a-d and e,f are each one canvas in the original producers, so each is regenerated and checked as one canvas.

In [ ]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered panels; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import panels
fl.init()
from IPython.display import Image, display

## Panels a-d

a, fraction of distinct queries that match at least one reservoir protein, by length (9-50 residues), for the authentic predictions and five composition-preserving scrambles (seeds 20260901, 20260913-20260916); the inset enlarges the scrambles at 9-13 residues. b, number of distinct authentic queries per length (the denominator). c,d, median accepted peptides (with exclusive quartiles) and median selected proteins over 58 acquisitions for minimum lookup lengths 6-12 at top 30% (c) and top-scoring fractions 5-100% at minimum length nine (d).

Claims: "The fraction of scrambled queries matching a reservoir protein fell from 8.1–8.9% for nine-residue queries to 0.7–1.1% at ten residues, with no match in 86,000 query–seed evaluations at ≥13 residues" (v28d line 59); "Selection sensitivity used the same 58 healthy-stool acquisitions for 19 settings" (v28d line 679).

n and unit: a,b one acquisition, distinct query (sequence) within arm and seed; c,d 58 acquisitions per arm, the same acquisitions in every arm. Test: none (descriptive fractions, medians and exclusive quartiles).

In [ ]:
panels.panel_abcd(SD, OUT / "panel_abcd.pdf")
print(panels.compose_abcd(OUT / "panel_abcd.pdf", OUT / "SFig02_v28f.pdf", OUT / "SFig02_v28f.png"))
import fitz
fitz.open(OUT / "panel_abcd.pdf")[0].get_pixmap(dpi=300, alpha=False).save(OUT / "panel_abcd.png")
display(Image(filename=str(OUT / "SFig02_v28f.png"), width=700))
for k, v in panels.stat_ab_lookup(SD).items():
    print(k, v)
for k, v in panels.stat_cd_grid(SD).items():
    print(k, v)

## Panels e,f

e, gained (blue) and lost (vermilion) peptide-acquisition identifications of each relaxed-lookup arm relative to exact lookup, summed over 48 acquisitions (15 CAMPI, 32 healthy stool, 1 CapScan) after within-acquisition deduplication at nominal peptide q≤0.01; the right column is the net. f, of 215 gained composition-preserving rearrangement pairs in six acquisitions, 193 had no diagnostic annotated fragment ion and 22 had at least one.

Claims: "admitting recorded bags gained 6,961 and lost 5,941 peptide–acquisition identifications relative to exact lookup (net +1,020)" and "In six acquisitions, 193 of 215 gained identifications ... had no annotated fragment ion distinguishing the two orders" (v28d line 67).

n and unit: e 48 acquisitions per arm, counts of peptide-acquisition identifications; f 215 same-spectrum comparison pairs in six acquisitions. Test: none (descriptive counts; the legend states no inferential error bars).

In [ ]:
panels.panel_ef(SD, OUT / "panel_ef.pdf")
print(panels.compose_ef(OUT / "panel_ef.pdf", OUT / "SFig02_continued_v28f.png"))
fitz.open(OUT / "panel_ef.pdf")[0].get_pixmap(dpi=300, alpha=False).save(OUT / "panel_ef.png")
display(Image(filename=str(OUT / "SFig02_continued_v28f.png"), width=700))
for k, v in {**panels.stat_e_bags(SD), **{'fragment_' + k: v for k, v in panels.stat_f_fragment(SD).items()}}.items():
    print(k, v)

## v30 house style (panels a-f)

The v30 figure redraws both canvases from the same tables in the house style: Söhne type, ink, Okabe-Ito colours by meaning (authentic and accepted peptides blue, scrambled and selected proteins orange, gained blue, lost vermilion), drawn at the placed widths (160 mm for a-d, 155 mm for e,f). Subpanel titles are removed (their content is in the legend), the net values of e become right-hand tick labels, and the ≤/≥ labels are composed from Söhne glyphs. `build_v30` first runs the producers above with saving intercepted and asserts that every plotted value, axis limit and scale is identical.


In [ ]:
v30 = panels.build_v30(SD, OUT)
print("v30 values identical to producer: a-d", v30["values_identical_abcd"], "| e,f", v30["values_identical_ef"])
print("label strings lost (titles and composed labels only):", v30["label_strings_lost_abcd"], v30["label_strings_lost_ef"])
print("png px:", v30["png_px"])
display(Image(filename=str(OUT / "SFig02_v30.png"), width=700))
display(Image(filename=str(OUT / "SFig02_continued_v30.png"), width=700))


## Pixel checks and numbers

Each regenerated canvas is compared with the original producer's output: the vector PDFs at 300 dpi, and the two v28f images at their own resolution. The two v30 house-style images (Söhne text) are compared with the v30 PNGs at 300 dpi. Both producers used DejaVu Sans, which ships with matplotlib; the check is still gated on `FASTALAKE_FONT_DIR` as for every unit. Then every number in `NUMBERS.tsv` is recomputed and printed in file order.

In [ ]:
import os, csv, re
if os.environ.get("FASTALAKE_FONT_DIR") or os.environ.get("FL_STYLE_DIR"):
    from tools.check_pixels import compare
    for name in ["panel_abcd.pdf", "SFig02_v28f.png", "panel_ef.pdf", "SFig02_continued_v28f.png",
                 "SFig02_v30.png", "SFig02_continued_v30.png"]:
        r = compare(OUT / name, UNIT / "expected" / name, UNIT / "checks" / name.split(".")[0], dpi=300)
        print(name, r["verdict"], r.get("differing_pixels"))
else:
    print("pixel checks SKIPPED: fonts absent")

cache = {}
with open(UNIT / "NUMBERS.tsv", encoding="utf-8") as fh:
    for row in csv.DictReader(fh, delimiter="\t"):
        if row["status"] == "OPEN":
            print(row["v28d_line"], row["printed_value"], "OPEN")
            continue
        fn, key = re.match(r".*:(\w+)\[(\w+)\]$", row["code"]).groups()
        if fn not in cache:
            cache[fn] = getattr(panels, fn)(SD)
        v = cache[fn][key]
        v = repr(v) if isinstance(v, float) else str(v)
        assert v == row["recomputed_value"], (row, v)
        print(row["v28d_line"], row["printed_value"], v, row["status"])

## Current 717-acquisition continuation
The cells above retain the historical pilot. These panels and counts reproduce v42.

In [ ]:
from figures.SFig02 import current
current_build = current.render(SD, OUT)
print(current.numbers(SD))
display(Image(filename=str(current_build["png"]), width=700))
if fl._HAVE_STYLE:
    from tools.check_pixels import compare
    for ext in ("png", "pdf"):
        r = compare(current_build[ext], UNIT / "expected" / f"SFig02_continued_v42.{ext}", UNIT / "checks" / f"SFig02_continued_v42_{ext}", dpi=300)
        print(r["verdict"], r.get("differing_pixels"))
        assert r["verdict"] == "IDENTICAL", r
else:
    print("current pixel checks SKIPPED: fonts absent")
